# Cardiovascular Risk Screening for Kenyans Using CDC BRFSS Data

---

## 1. Introduction: Business Understanding

### The Real-World Problem

Cardiovascular disease—including heart disease and heart attack—is a growing 
public health crisis in Kenya, driven by rapid urbanization, dietary shifts 
toward processed foods, rising obesity, and limited access to preventive 
screening. Most Kenyans interact with the health system only when acutely ill; 
opportunistic CVD risk screening is rare, especially at the community level.

### Stakeholders and How They Would Use This Model

**Kenya Ministry of Health — NCD Division**
Use the model's global feature importance rankings to prioritize which risk 
factors to target in the national NCD strategy. If smoking and hypertension 
rank highest, allocate more budget to tobacco control and BP screening.

**County Health Departments (47 counties)**
Use the deployed risk calculator to support county-level screening drives. 
Identify which populations have the highest predicted risk profiles.

**Healthcare Providers (hospitals, clinics, community health units)**
Use the web-based risk calculator during routine visits as an opportunistic 
screening aid — input self-reported risk factors, receive a risk score + 
explanation to guide clinical conversation.

**Community Health Volunteers (CHVs)**
Use the simplified risk calculator during household visits. Flag high-risk 
individuals for referral without needing lab equipment.

**NHIF/SHA and Private Health Insurers**
Use population-level risk driver analysis to design preventive wellness 
benefits targeting factors with the highest attributable risk.

### Our Solution
We build an interpretable ML classifier that predicts whether an individual 
has heart disease or a prior heart attack based on self-reported behavioral 
and clinical indicators — enabling stakeholders to screen thousands of 
individuals and power automated risk stratification dashboards.

## 2. Data Understanding

### Data Source

The dataset comes from the **CDC BRFSS (Behavioral Risk Factor Surveillance 
System)** — the world's largest ongoing telephone health survey, collecting 
data on health-related risk behaviors, chronic conditions, and use of 
preventive services from ~400,000+ U.S. adults annually.

Used as a **methodological proxy** for Kenya: the risk factors captured 
(smoking, BMI, physical inactivity, diabetes, hypertension) are the same 
modifiable factors driving CVD in Kenya.

### Why This Dataset Is Suitable

| Criterion | Assessment |
|-----------|-----------|
| **Labeled data** | Self-reported heart disease/heart attack enables supervised classification |
| **Real-world indicators** | Behavioral + clinical self-report variables match what CHVs can collect |
| **Sufficient size** | 90,000+ respondents — large enough for ML modeling |
| **Relevant risk factors** | Smoking, BMI, diabetes, hypertension, physical activity — all in Kenya STEPwise |
| **Manageable** | Computationally feasible on local machines |

### Data Limitations (Identified Upfront)

1. **US population** — demographics, healthcare access, and disease 
   prevalence differ from Kenya. Requires recalibration for Kenyan use.
2. **Self-reported outcomes** — relies on diagnosis awareness; undiagnosed 
   cases are missed.
3. **Cross-sectional** — cannot establish causality or temporal sequence.
4. **No objective measurements** — no BP readings, lab values, or ECG.
5. **Class imbalance** — CVD prevalence ~10%, making minority class 
   prediction challenging.

In [6]:
# =============================================================================
# 3. Data Preparation
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load raw data
df = pd.read_sas("LLCP2024.XPT")
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Columns: 301 entries, _STATE to _AIDTST4
dtypes: float64(296), object(5)
memory usage: 1.0+ GB
None
              _STATE         FMONTH       DISPCODE          _PSU  CTELENM1  \
count  457670.000000  457670.000000  457670.000000  4.576700e+05   81730.0   
mean       30.823537       6.426995    1118.669784  2.024007e+09       1.0   
std        15.939364       3.356938      38.966920  7.488291e+03       0.0   
min         1.000000       1.000000    1100.000000  2.024000e+09       1.0   
25%        18.000000       4.000000    1100.000000  2.024002e+09       1.0   
50%        31.000000       7.000000    1100.000000  2.024005e+09       1.0   
75%        44.000000       9.000000    1100.000000  2.024008e+09       1.0   
max        78.000000      12.000000    1200.000000  2.024044e+09       1.0   

           PVTRESD1  COLGHOUS  STATERE1  CELPHON1       LADULT1  ...  \
count  81730.000000      14.0   81730.0   81730

In [10]:
df.columns.tolist()

['_STATE',
 'FMONTH',
 'IDATE',
 'IMONTH',
 'IDAY',
 'IYEAR',
 'DISPCODE',
 'SEQNO',
 '_PSU',
 'CTELENM1',
 'PVTRESD1',
 'COLGHOUS',
 'STATERE1',
 'CELPHON1',
 'LADULT1',
 'NUMADULT',
 'RESPSLC1',
 'LANDSEX3',
 'SAFETIME',
 'CTELNUM1',
 'CELLFON5',
 'CADULT1',
 'CELLSEX3',
 'PVTRESD3',
 'CCLGHOUS',
 'CSTATE1',
 'LANDLINE',
 'HHADULT',
 'SEXVAR',
 'GENHLTH',
 'PHYSHLTH',
 'MENTHLTH',
 'POORHLTH',
 'PRIMINS2',
 'PERSDOC3',
 'MEDCOST1',
 'CHECKUP1',
 'EXERANY2',
 'LASTDEN4',
 'RMVTETH4',
 'CVDINFR4',
 'CVDCRHD4',
 'CVDSTRK3',
 'ASTHMA3',
 'ASTHNOW',
 'CHCSCNC1',
 'CHCOCNC1',
 'CHCCOPD3',
 'ADDEPEV3',
 'CHCKDNY2',
 'HAVARTH4',
 'DIABETE4',
 'DIABAGE4',
 'MARITAL',
 'EDUCA',
 'RENTHOM1',
 'NUMHHOL4',
 'NUMPHON4',
 'CPDEMO1C',
 'VETERAN3',
 'EMPLOY1',
 'CHILDREN',
 'INCOME3',
 'PREGNANT',
 'WEIGHT2',
 'HEIGHT3',
 'DEAF',
 'BLIND',
 'DECIDE',
 'DIFFWALK',
 'DIFFDRES',
 'DIFFALON',
 'HADMAM',
 'HOWLONG',
 'CERVSCRN',
 'CRVCLCNC',
 'CRVCLPAP',
 'CRVCLHPV',
 'HADHYST2',
 'HADSIGM4',
 'COLNSIGM',

In [8]:
df

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_LCSCTSN,_LCSPSTF,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK3,_RFDRHV9,_FLSHOT7,_PNEUMO3,_AIDTST4
0,1.0,2.0,b'02282024',b'02',b'28',b'2024',1100.0,b'2024000001',2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,2.0,2.0
1,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000002',2.024000e+09,1.0,...,4.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
2,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000003',2.024000e+09,1.0,...,4.0,2.0,1.0,1.000000e+02,2.0,1.400000e+03,1.0,NaN,NaN,2.0
3,1.0,2.0,b'02282024',b'02',b'28',b'2024',1100.0,b'2024000004',2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
4,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000005',2.024000e+09,1.0,...,3.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457665,78.0,7.0,b'07122024',b'07',b'12',b'2024',1200.0,b'2024001344',2.024001e+09,NaN,...,NaN,NaN,9.0,9.000000e+02,9.0,9.990000e+04,9.0,9.0,9.0,NaN
457666,78.0,7.0,b'07182024',b'07',b'18',b'2024',1100.0,b'2024001345',2.024001e+09,NaN,...,4.0,2.0,1.0,1.000000e+02,9.0,3.500000e+03,2.0,2.0,2.0,1.0
457667,78.0,7.0,b'07172024',b'07',b'17',b'2024',1100.0,b'2024001346',2.024001e+09,NaN,...,4.0,9.0,1.0,7.000000e+00,1.0,1.400000e+02,1.0,1.0,1.0,2.0
457668,78.0,7.0,b'07202024',b'07',b'20',b'2024',1200.0,b'2024001347',2.024001e+09,NaN,...,NaN,NaN,9.0,9.000000e+02,9.0,9.990000e+04,9.0,NaN,NaN,NaN


In [11]:
# =============================================================================
# FINAL: Column selection for BRFSS 2024 CVD project
# =============================================================================

keep = [
    # Target (pre-computed by CDC: 1=Yes MI or CHD, 2=No)
    '_MICHD',       # Calculated: heart attack OR coronary heart disease

    # Demographics
    'SEXVAR',       # Sex
    '_AGE80',       # Age (top-coded at 80)
    '_IMPRACE',     # Imputed race/ethnicity

    # Behavioral (modifiable)
    'SMOKE100',     # Smoked 100+ cigarettes
    '_SMOKER3',     # Computed smoking status
    'EXERANY2',     # Any exercise past 30 days
    '_TOTINDA',     # Computed: any physical activity (backup for EXERANY2)
    'ALCDAY4',      # Days per week/month drinking alcohol
    '_RFDRHV9',     # Computed: heavy drinker

    # Clinical (self-reported)
    'DIABETE4',     # Diabetes
    'CVDSTRK3',     # Stroke (comorbidity signal)
    'CHCCOPD3',     # COPD
    'ADDEPEV3',     # Depression
    'CHCKDNY2',     # Kidney disease
    'HAVARTH4',     # Arthritis
    'DIFFWALK',     # Difficulty walking

    # Body composition
    '_BMI5',        # BMI (x100)
    '_BMI5CAT',     # BMI category (1=Underweight, 2=Normal, 3=Overweight, 4=Obese)

    # General health
    'GENHLTH',      # General health (1-5)
    'PHYSHLTH',     # Physical health days (past 30)
    'MENTHLTH',     # Mental health days (past 30)
    '_RFHLTH',      # Computed: good or better health

    # Socioeconomic
    'INCOME3',      # Income
    'EDUCA',        # Education
    '_EDUCAG',      # Computed education level
]

available = [col for col in keep if col in df.columns]
missing  = [col for col in keep if col not in df.columns]

print(f"✅ Found {len(available)}/{len(keep)} columns")
if missing:
    print(f"❌ Missing: {missing}")

df_selected = df[available].copy()
print(f"Shape: {df_selected.shape}")
df_selected.head()

✅ Found 26/26 columns
Shape: (457670, 26)


,_MICHD,SEXVAR,_AGE80,_IMPRACE,SMOKE100,_SMOKER3,EXERANY2,_TOTINDA,ALCDAY4,_RFDRHV9,...,DIFFWALK,_BMI5,_BMI5CAT,GENHLTH,PHYSHLTH,MENTHLTH,_RFHLTH,INCOME3,EDUCA,_EDUCAG
0,2.0,2.0,78.0,1.0,2.0,4.0,1.0,1.0,888.0,1.0,...,2.0,2249.0,2.0,3.0,2.0,88.0,1.0,99.0,4.0,2.0
1,1.0,1.0,80.0,1.0,1.0,3.0,1.0,1.0,888.0,1.0,...,2.0,2583.0,3.0,1.0,88.0,88.0,1.0,11.0,6.0,4.0
2,2.0,1.0,59.0,1.0,1.0,1.0,1.0,1.0,230.0,1.0,...,2.0,2253.0,2.0,2.0,30.0,88.0,1.0,99.0,5.0,3.0
3,2.0,1.0,80.0,1.0,2.0,4.0,1.0,1.0,888.0,1.0,...,2.0,2509.0,3.0,1.0,88.0,88.0,1.0,6.0,6.0,4.0
4,2.0,1.0,47.0,1.0,2.0,4.0,2.0,2.0,888.0,1.0,...,1.0,1977.0,2.0,3.0,88.0,88.0,1.0,3.0,5.0,3.0


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Columns: 301 entries, _STATE to _AIDTST4
dtypes: float64(296), object(5)
memory usage: 1.0+ GB


In [14]:
df_selected.describe()

,_MICHD,SEXVAR,_AGE80,_IMPRACE,SMOKE100,_SMOKER3,EXERANY2,_TOTINDA,ALCDAY4,_RFDRHV9,...,DIFFWALK,_BMI5,_BMI5CAT,GENHLTH,PHYSHLTH,MENTHLTH,_RFHLTH,INCOME3,EDUCA,_EDUCAG
count,452464.000000,457670.000000,457670.000000,457670.000000,428810.000000,457670.000000,457667.000000,457670.000000,418449.000000,457670.000000,...,438998.000000,414633.000000,414633.000000,457665.000000,457665.000000,457667.000000,457670.000000,448401.000000,457663.000000,457670.000000
mean,1.906428,1.524795,55.084784,1.788977,1.643007,3.810357,1.251427,1.255236,528.701603,1.869063,...,1.860248,2855.680093,3.009244,2.659362,57.918113,57.728739,1.219816,21.215446,5.043976,3.077680
std,0.291233,0.499385,18.127835,1.526311,0.667560,1.657696,0.547624,0.592447,358.151145,2.414067,...,0.525955,658.616131,0.836549,1.081163,37.872634,38.072038,0.576010,31.355378,1.053186,1.043475
min,1.000000,1.000000,18.000000,1.000000,1.000000,1.000000,1.000000,1.000000,100.000000,1.000000,...,1.000000,1200.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,1.000000,40.000000,1.000000,1.000000,3.000000,1.000000,1.000000,202.000000,1.000000,...,2.000000,2414.000000,2.000000,2.000000,12.000000,10.000000,1.000000,6.000000,4.000000,2.000000
50%,2.000000,2.000000,58.000000,1.000000,2.000000,4.000000,1.000000,1.000000,230.000000,1.000000,...,2.000000,2744.000000,3.000000,3.000000,88.000000,88.000000,1.000000,8.000000,5.000000,3.000000
75%,2.000000,2.000000,71.000000,2.000000,2.000000,4.000000,1.000000,1.000000,888.000000,1.000000,...,2.000000,3175.000000,4.000000,3.000000,88.000000,88.000000,1.000000,10.000000,6.000000,4.000000
max,2.000000,2.000000,80.000000,6.000000,9.000000,9.000000,9.000000,9.000000,999.000000,9.000000,...,9.000000,9984.000000,4.000000,9.000000,99.000000,99.000000,9.000000,99.000000,9.000000,9.000000
